# Connecting Budget Bill to Fiscal Data
These codes in this file aim to connect Budget Bill to Fiscal Data in fiscal year 2023 (July 2023 - June 2024)

In [11]:
import pandas as pd
import numpy as np
import glob
import os

## 1. Implement by using FY2023P01 only

#### A. Define the function

In [12]:
def pad_code(val, length):
    if pd.isna(val) or val == '' or str(val).lower() == 'nan':
        return '0'.zfill(length)
    try:
        clean_val = str(int(float(val)))
        return clean_val.zfill(length)
    except:
        return str(val).zfill(length)

def get_final_bill_code(row):
    obj = str(row.get('object_code', '')).replace('.0', '').strip()
    proj = str(row.get('program_code', '')).replace('.0', '').strip()
    target = obj if (obj and obj not in ['0', 'nan', 'EMPTY']) else proj
    return target if target else '0'


#### B. Prepare the Budget bill

In [13]:
Bill = "2023_Budget_Final_Amounts.csv"
bill_df = pd.read_csv(Bill)

print("The first few rows of the bill dataframe are:")
print(bill_df.head())

# Extract relevant columns and create new DataFrame for processing
bill_df_new = bill_df[["item_number", "program_code", "object_code", "revised_amount"]].copy()

# Extract business_unit, budget_reference, fund_code from item_number
bill_df_new[["business_unit", "budget_reference", "fund_code"]] = bill_df_new["item_number"].str.split("-", expand=True)

# Padding for business_unit, budget_reference, fund_code
bill_df_new['clean_bu'] = bill_df_new['business_unit'].apply(lambda x: pad_code(x, 4))
bill_df_new['clean_ref'] = bill_df_new['budget_reference'].apply(lambda x: pad_code(x, 3))
bill_df_new['clean_fund'] = bill_df_new['fund_code'].apply(lambda x: pad_code(x, 4))

# Create Budget_ID using the cleaned components and the final bill code logic
bill_df_new['Budget_ID'] = (
    "2023-" + 
    bill_df_new['clean_bu'] + "-" + 
    bill_df_new['clean_ref'] + "-" + 
    bill_df_new['clean_fund'] + "-" + 
    bill_df_new.apply(get_final_bill_code, axis=1)
)

# Again, show the first few rows to verify the new Budget_ID column
print("\nThe first few rows of the processed bill dataframe with Budget_ID are:")
print(bill_df_new.head())

The first few rows of the bill dataframe are:
     item_number program_code object_code  fund_code  revised_amount
0  0110-001-0001        960.0    101001.0        1.0        -6751000
1  0110-001-0001        960.0    317292.0        1.0        -1712000
2  0110-001-0001        960.0    317295.0        1.0          -11000
3  0110-001-0001        960.0    500004.0        1.0      -168851000
4  0110-001-0001        960.0       EMPTY        1.0       177325000

The first few rows of the processed bill dataframe with Budget_ID are:
     item_number program_code object_code  revised_amount business_unit  \
0  0110-001-0001        960.0    101001.0        -6751000          0110   
1  0110-001-0001        960.0    317292.0        -1712000          0110   
2  0110-001-0001        960.0    317295.0          -11000          0110   
3  0110-001-0001        960.0    500004.0      -168851000          0110   
4  0110-001-0001        960.0       EMPTY       177325000          0110   

  budget_referenc

#### C. Process fiscal data and merge with bill data

In [14]:
Fiscal = "Vendor_FY23P01.csv"
temp_fiscal = pd.read_csv(Fiscal, low_memory=False)

# Limit to FY23 data
temp_fiscal = temp_fiscal[temp_fiscal['year_of_enactment'] == 2023].copy()

# Padding for business_unit, budget_reference, fund_code in fiscal data
temp_fiscal['clean_bu'] = temp_fiscal['business_unit'].apply(lambda x: pad_code(x, 4))
temp_fiscal['clean_ref'] = temp_fiscal['budget_reference'].apply(lambda x: pad_code(x, 3))
temp_fiscal['clean_fund'] = temp_fiscal['fund_code'].apply(lambda x: pad_code(x, 4))

# Create Budget_ID_7 and Budget_ID_4 for matching
temp_fiscal['prog_str'] = temp_fiscal['program_code'].astype(str).str.replace('.0', '').str.strip()
temp_fiscal['Budget_ID_7'] = "2023-" + temp_fiscal['clean_bu'] + "-" + temp_fiscal['clean_ref'] + "-" + temp_fiscal['clean_fund'] + "-" + temp_fiscal['prog_str'].str[:7]
temp_fiscal['Budget_ID_4'] = "2023-" + temp_fiscal['clean_bu'] + "-" + temp_fiscal['clean_ref'] + "-" + temp_fiscal['clean_fund'] + "-" + temp_fiscal['prog_str'].str[:4]

# --- Execution of Matching ---
# Step 1: Object code level(Match 7 digits)
m1 = pd.merge(bill_df_new, temp_fiscal, left_on='Budget_ID', right_on='Budget_ID_7', how='inner', suffixes=('', '_f'))

# Step 2: Program code level (Match 4 digits) for unmatched records from Step 1
matched_ids = m1['Budget_ID'].unique()
unmatched_bill = bill_df_new[~bill_df_new['Budget_ID'].isin(matched_ids)]
m2 = pd.merge(unmatched_bill, temp_fiscal, left_on='Budget_ID', right_on='Budget_ID_4', how='inner', suffixes=('', '_f'))

# Combine matched results from both steps
p01_merged = pd.concat([m1, m2], ignore_index=True)

# マスター（予算）を左側に残して全行保持
final_report_p01 = pd.merge(bill_df_new, p01_merged, on='Budget_ID', how='left', suffixes=('', '_final'))

#### D. Output

In [15]:
# 1. Detail Report
detail_cols = [
    'Budget_ID', 
    'year_of_enactment', 
    'business_unit', 
    'agency_name', 
    'department_name',  
    'revised_amount', 
    'monetary_amount', 
    'VENDOR_NAME', 
    'accounting_date'
]
final_report_clean = final_report_p01[[c for c in detail_cols if c in final_report_p01.columns]].copy()
final_report_clean.to_csv("Detail_FY23P01.csv", index=False, encoding='utf-8-sig')

# 2. Crosswalk Table
crosswalk_cols = [
    'Budget_ID', 
    'agency_name',      
    'department_name',  
    'program_description'
]
crosswalk_p01 = final_report_p01[final_report_p01['monetary_amount'].notna()][
    [c for c in crosswalk_cols if c in final_report_p01.columns]
].drop_duplicates()
crosswalk_p01.to_csv("Crosswalk_FY23P01.csv", index=False, encoding='utf-8-sig')

print(f"P01 Processing Complete.")
print(f"Matched rows: {final_report_p01['monetary_amount'].notna().sum()}")

P01 Processing Complete.
Matched rows: 9061


## 2. Expand all fiscal data

In [16]:
# 1. Prepare an empty list to collect monthly merged dataframes
final_combined_list = []

# 2. Designate the path to the folder containing the actual expenditure data files (Vendor_FY2*P*.csv)
file_pattern = "Vendor_FY2*P*.csv" 
files = glob.glob(file_pattern)
files.sort()

print(f"Total {len(files)} files to process...")

for file in files:
    print(f"Processing: {file}")
    temp_fiscal = pd.read_csv(file, low_memory=False)

    temp_fiscal = temp_fiscal[temp_fiscal['year_of_enactment'] == 2023].copy()
    
    temp_fiscal['clean_bu'] = temp_fiscal['business_unit'].apply(lambda x: pad_code(x, 4))
    temp_fiscal['clean_ref'] = temp_fiscal['budget_reference'].apply(lambda x: pad_code(x, 3))
    temp_fiscal['clean_fund'] = temp_fiscal['fund_code'].apply(lambda x: pad_code(x, 4))
    
    temp_fiscal['prog_str'] = temp_fiscal['program_code'].astype(str).str.replace('.0', '').str.strip()
    temp_fiscal['Budget_ID_7'] = "2023-" + temp_fiscal['clean_bu'] + "-" + temp_fiscal['clean_ref'] + "-" + temp_fiscal['clean_fund'] + "-" + temp_fiscal['prog_str'].str[:7]
    temp_fiscal['Budget_ID_4'] = "2023-" + temp_fiscal['clean_bu'] + "-" + temp_fiscal['clean_ref'] + "-" + temp_fiscal['clean_fund'] + "-" + temp_fiscal['prog_str'].str[:4]


    m1 = pd.merge(bill_df_new, temp_fiscal, left_on='Budget_ID', right_on='Budget_ID_7', how='inner', suffixes=('', '_f'))
    matched_ids = m1['Budget_ID'].unique()
    unmatched_bill = bill_df_new[~bill_df_new['Budget_ID'].isin(matched_ids)]
    m2 = pd.merge(unmatched_bill, temp_fiscal, left_on='Budget_ID', right_on='Budget_ID_4', how='inner', suffixes=('', '_f'))
    
    final_combined_list.append(m1)
    final_combined_list.append(m2)

# 3. Combine all monthly dataframes into one big dataframe
full_year_df = pd.concat(final_combined_list, ignore_index=True)

# 4. Finally, merge with the original master to ensure all budget items are included, even those without any spending
final_report = pd.merge(bill_df_new, full_year_df, on='Budget_ID', how='left', suffixes=('', '_final'))

Total 30 files to process...
Processing: Vendor_FY23P01.csv
Processing: Vendor_FY23P02.csv
Processing: Vendor_FY23P03.csv
Processing: Vendor_FY23P04.csv
Processing: Vendor_FY23P05.csv
Processing: Vendor_FY23P06.csv
Processing: Vendor_FY23P07.csv
Processing: Vendor_FY23P08.csv
Processing: Vendor_FY23P09.csv
Processing: Vendor_FY23P10.csv
Processing: Vendor_FY23P11.csv
Processing: Vendor_FY23P12.csv
Processing: Vendor_FY24P01.csv
Processing: Vendor_FY24P02.csv
Processing: Vendor_FY24P03.csv
Processing: Vendor_FY24P04.csv
Processing: Vendor_FY24P05.csv
Processing: Vendor_FY24P06.csv
Processing: Vendor_FY24P07.csv
Processing: Vendor_FY24P08.csv
Processing: Vendor_FY24P09.csv
Processing: Vendor_FY24P10.csv
Processing: Vendor_FY24P11.csv
Processing: Vendor_FY24P12.csv
Processing: Vendor_FY25P01.csv
Processing: Vendor_FY25P02.csv
Processing: Vendor_FY25P03.csv
Processing: Vendor_FY25P04.csv
Processing: Vendor_FY25P05.csv
Processing: Vendor_FY25P06.csv


In [17]:
# --- 1. Order the columns ---
detail_cols = [
    'Budget_ID', 
    'year_of_enactment', 
    'business_unit', 
    'agency_name', 
    'department_name',  
    'revised_amount', 
    'monetary_amount', 
    'VENDOR_NAME', 
    'accounting_date'
]
final_report_clean = final_report[[c for c in detail_cols if c in final_report.columns]].copy()

# Save as a csv file for the final report before aggregation
final_report_clean.to_csv("Final_Budget_Expenditure_Details.csv", index=False, encoding='utf-8-sig')

# --- 2. Create summary report ---
summary_report = final_report_clean.groupby(['Budget_ID', 'year_of_enactment']).agg({
    'revised_amount': 'first',      
    'monetary_amount': 'sum',       
    'agency_name': 'first'          
}).reset_index()

# Calculate Spending Rate
summary_report['Spending_Rate'] = (summary_report['monetary_amount'] / summary_report['revised_amount']) * 100

# --- 3. Save the summary report to a new CSV file
summary_report.to_csv("Budget_Spending_Summary_FY23-25.csv", index=False, encoding='utf-8-sig')

print("Complete! The summary file (Budget_Spending_Summary_FY23-25.csv) has been created.")

Complete! The summary file (Budget_Spending_Summary_FY23-25.csv) has been created.


In [18]:
# Check the number of rows in final_report_clean and summary_report
print(f"Final Report Rows: {len(final_report_clean)}")
print(f"Summary Report Rows: {len(summary_report)}")

Final Report Rows: 1453941
Summary Report Rows: 928


### Make a Crosswalk table 

In [ ]:
crosswalk_df = final_report[final_report['monetary_amount'].notna()].copy()

crosswalk_cols = [
    'Budget_ID', 
    'agency_name',      
    'department_name',  
    'program_description'
]

crosswalk_table = crosswalk_df[[c for c in crosswalk_cols if c in crosswalk_df.columns]].drop_duplicates()

crosswalk_table.to_csv("Final_Crosswalk_Table_FY23-25.csv", index=False, encoding='utf-8-sig')

print(f"Complete! The crosswalk table (Final_Crosswalk_Table_FY23-25.csv) has been created.")
print(f"Valid pairs: {len(crosswalk_table)}")

Complete! The crosswalk table (Final_Crosswalk_Table_FY23-25.csv) has been created.
Valid pairs: 1122
